### Why does each process need its own Kernel Stack?

When the kernel runs C code, it needs two things:

1. A properly configured page table to access memory.
2. A valid stack pointer (`sp`) to execute functions. (needs a stack because number of registers is not enough, remember?)

Different user processes make system calls, asking the kernel to perform different tasks.

However, the kernel **may not finish one task before switching to another**.

For example:

1. Process A calls `read()`.
2. The kernel starts executing `sys_read()`.
3. The requested data isn't ready, so Process A goes to sleep.
4. The kernel switches to Process B.
5. Later, Process A wakes up and must resume exactly where it left off inside the kernel.

**The key problem: Process A's kernel function calls haven't returned yet!**

Their stack frames must be preserved while Process B runs.

Therefore, each process needs its own Kernel Stack to preserve its unfinished kernel call chain.

The kernel also saves the process's execution context (including `sp` and `ra`) so it can resume execution later.

When a user process enters the kernel through a trap, xv6's trampoline code loads that process's Kernel Stack Pointer into `sp`.

**Mental Model:**

- **Kernel Page Table:** Where can the kernel access memory?
- **Kernel Stack:** Where does this process keep its active kernel function frames?
- **Saved Context:** Where and how should this process resume execution?

Different processes can share the same Kernel Page Table while maintaining independent Kernel Stacks.

---

### What stack kernel use when start? there is no process yet!

At kernel/entry.S, already set a Boot Stack:
>la sp, stack0

stack0 is defined in kernel/start.c:
```c
__attribute__((aligned(16)))
char stack0[4096 * NCPU];
```

whole flow:

```text
CPU boots
    |
    v
entry.S
    | Set sp = Boot Stack
    v
start()
    |
    v
main()
    | Initialize Kernel
    | Create first Process
    v
scheduler()
    | Save scheduler sp
    | Load Process A sp
    v
Process A Kernel Stack
    | Process sleeps or yields
    | Save Process A sp
    | Restore scheduler sp
    v
Scheduler Stack (original Boot Stack!)
    |
    | Choose Process B
    v
Process B Kernel Stack
```